# 5 skyrius. Eksperimentų rezultatai ir jų analizė
## 5.1 poskyris. Modelių atranka

Šis Jupyter Notebook failas generuoja akademines diagramas bakalauro darbo 5.1 poskyriui.  
Tema: kalbos triukšmo šalinimas naudojant CNN tipo neuroninius tinklus.

**Duomenų šaltinis:** `results/all_runs.csv`  
**Atrankos eksperimentai:** `screen_lt_` prefikso eksperimentai (LIEPA duomenų rinkinys)

---

## 1. Importai ir nustatymai

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import os, warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.grid':         True,
    'grid.linestyle':    '--',
    'grid.alpha':        0.4,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

def setup_ax(ax):
    ax.yaxis.grid(False)
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

CSV_PATH    = os.path.join('results', 'all_runs.csv')
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

REF_COLOR = '#C0392B'
MODELS_ORDER = [
    'BaseCNN', 'MaskCNN', 'DilatedCNN', 'DilatedMaskCNN',
    'CNNAutoencoder', 'UNetCNN', 'UNetMaskCNN', 'UNetDilatedMaskCNN'
]
LR_STYLES = {
    0.001:  {'color': '#444444', 'hatch': '////', 'edgecolor': '#222222'},
    0.0005: {'color': '#BBBBBB', 'hatch': '',     'edgecolor': '#555555'},
}
BAR_H  = 0.34
OFFSET = 0.19

print('Importai sėkmingi.')
print(f'Paveikslų katalogas: {os.path.abspath(FIGURES_DIR)}')

---
## 2. Duomenų nuskaitymas

In [ ]:
df_all = pd.read_csv(CSV_PATH)
print(f'Iš viso eilučių CSV faile: {len(df_all)}')

mask_screen   = df_all['experiment_id'].str.startswith('screen_lt_')
df_screen_raw = df_all[mask_screen].copy()
print(f'screen_lt_ eilučių kiekis (prieš dublikatų šalinimą): {len(df_screen_raw)}')
print('Unikalūs modeliai:',   sorted(df_screen_raw['model_name'].unique()))
print('Nuostolių funkcijos:', sorted(df_screen_raw['loss_name'].unique()))
print('Mokymosi greičiai:',   sorted(df_screen_raw['learning_rate'].unique()))

---
## 3. Dublikatų tikrinimas ir valymas

In [ ]:
dup_mask = df_screen_raw.duplicated(subset=['experiment_id'], keep=False)
df_dups  = df_screen_raw[dup_mask]
if len(df_dups):
    print(f'Rasta dublikatų ({len(df_dups)} eilutės):')
    print(df_dups[['experiment_id','batch_size','learning_rate',
                   'best_val_loss','mean_pesq_enhanced']].to_string())
    print('Sprendimas: paliekamas paskutinis įrašas (batch_size=256).')
else:
    print('Dublikatų nerasta.')

df_screen = df_screen_raw.drop_duplicates(subset=['experiment_id'], keep='last').copy()
df_screen = df_screen.reset_index(drop=True)
print(f'Po šalinimo: {len(df_screen)} konfigūracijų')
n_m, n_l, n_lr = df_screen['model_name'].nunique(), df_screen['loss_name'].nunique(), df_screen['learning_rate'].nunique()
print(f'{n_m} arch × {n_l} nuostolių funkcijos × {n_lr} mokymosi greičiai = {n_m*n_l*n_lr}')

---
## 4. Atskaitos reikšmės

In [ ]:
PESQ_REF = df_screen['mean_pesq_noisy'].mean()
STOI_REF = df_screen['mean_stoi_noisy'].mean()
print(f'PESQ atskaitos reikšmė: {PESQ_REF:.4f}  (std={df_screen["mean_pesq_noisy"].std():.4f})')
print(f'STOI atskaitos reikšmė: {STOI_REF:.4f}  (std={df_screen["mean_stoi_noisy"].std():.4f})')

neg_snr = df_screen[df_screen['mean_delta_snr_est'] < 0]
print(f'Konfigūracijų su neigiamu ΔSNR: {len(neg_snr)} iš 32')
if len(neg_snr):
    print(neg_snr[['experiment_id','model_name','loss_name',
                   'learning_rate','mean_delta_snr_est']].to_string(index=False))

def format_lr(lr): return '0,001' if lr == 0.001 else '0,0005'
df_screen['etikete'] = (df_screen['model_name'] + ' | ' +
                        df_screen['loss_name']   + ' | ' +
                        df_screen['learning_rate'].apply(format_lr))
best = df_screen.loc[df_screen['mean_pesq_enhanced'].idxmax()]
print(f'Geriausia konfigūracija (PESQ): {best["etikete"]}')
print(f'  PESQ={best["mean_pesq_enhanced"]:.3f}  STOI={best["mean_stoi_enhanced"]:.3f}  ΔSNR={best["mean_delta_snr_est"]:.2f} dB')

---
## 5. Diagramų funkcija

In [ ]:
# ─── Bendra horizontali stulpelinė diagrama ───────────────────────────────────
# fmt: 'pesq' | 'stoi' | 'delta' | 'val'
# higher_is_better=True  → geriausias (didžiausias) viršuje
# higher_is_better=False → geriausias (mažiausias) viršuje  (validavimo nuostolis)

def plot_bar(df_sub, metric_col, title, xlabel,
             fmt='pesq', ref_val=None, higher_is_better=True):

    arch_repr = {a: (df_sub[df_sub['model_name']==a][metric_col].max()
                     if higher_is_better else
                     df_sub[df_sub['model_name']==a][metric_col].min())
                 for a in MODELS_ORDER if a in df_sub['model_name'].values}

    # Ascending sort: mažiausia reikšmė bottom → didžiausia top
    # Validavimo nuostolis: reverse=True → didžiausias (blogiausias) bottom
    arch_sorted = sorted(arch_repr, key=arch_repr.get, reverse=(not higher_is_better))
    n = len(arch_sorted)

    all_vals          = df_sub[metric_col].tolist()
    x_min_d, x_max_d = min(all_vals), max(all_vals)
    x_rng             = max(x_max_d - x_min_d, 1e-6)

    if fmt == 'pesq':
        # Nuo 0 – stulpeliai pilni, skirtumai matomi (range ~0,5)
        x_left, lbl_pad = 0.0, x_max_d * 0.011
        x_right = x_max_d + lbl_pad * 12
    elif fmt == 'stoi':
        # STOI intervalas labai siauras (~0,014); xlim priartina, stulpeliai prasideda nuo 0
        x_left  = max(0.0, x_min_d - x_rng * 2.5)
        lbl_pad = x_rng * 0.15
        x_right = x_max_d + lbl_pad * 12
    elif fmt == 'delta':
        lbl_pad = x_rng * 0.04
        x_left  = min(x_min_d - x_rng * 0.08, 0.0) - lbl_pad * 3
        x_right = x_max_d + x_rng * 0.08 + lbl_pad * 6
    else:  # val
        x_left, lbl_pad = 0.0, x_max_d * 0.013
        x_right = x_max_d + lbl_pad * 12

    fig, ax = plt.subplots(figsize=(7.5, 6.8))
    ax.set_xlim(left=x_left, right=x_right)

    for yi, arch in enumerate(arch_sorted):
        for lr in [0.0005, 0.001]:
            r = df_sub[(df_sub['model_name']==arch) & (df_sub['learning_rate']==lr)]
            if not len(r): continue
            val = r[metric_col].values[0]
            sty = LR_STYLES[lr]
            off = OFFSET if lr == 0.001 else -OFFSET
            # barh width=val → stulpelis nuo 0 iki val (xlim kontroliuoja matymą)
            ax.barh(yi + off, val, height=BAR_H,
                    color=sty['color'], hatch=sty['hatch'],
                    edgecolor=sty['edgecolor'], linewidth=0.5, zorder=2)
            if fmt == 'pesq':    lbl = f'{val:.3f}'.replace('.', ',')
            elif fmt == 'stoi':  lbl = f'{val:.4f}'.replace('.', ',')
            elif fmt == 'delta': lbl = f'{val:.2f}'.replace('.', ',')
            else:                lbl = f'{val:.5f}'.replace('.', ',')
            if fmt == 'delta' and val < 0:
                ax.text(val - lbl_pad, yi + off, lbl, va='center', ha='right', fontsize=8, color='#222222')
            else:
                ax.text(val + lbl_pad, yi + off, lbl, va='center', ha='left',  fontsize=8, color='#222222')

    if fmt == 'delta':
        ax.axvline(x=0, color='#555555', linewidth=0.8, alpha=0.7, zorder=1)

    if ref_val is not None:
        ann_pad = lbl_pad * 0.3
        ax.axvline(x=ref_val, color=REF_COLOR, linestyle='--', linewidth=1.6, zorder=3)
        ax.text(ref_val + ann_pad, n - 0.4,
                f'{ref_val:.3f}'.replace('.', ','),
                color=REF_COLOR, fontsize=8.5, fontweight='bold', va='top', ha='left')

    ax.set_yticks(range(n)); ax.set_yticklabels(arch_sorted, fontsize=9.5)
    ax.set_xlabel(xlabel, fontsize=10); ax.set_ylabel('Architektūra', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=9)
    ax.set_ylim(-0.65, n - 0.35); setup_ax(ax)

    # Legenda PO diagrama — ne viduje
    handles = _leg_handles(ref_val is not None)
    ncol    = 3 if ref_val is not None else 2
    fig.legend(handles=handles, loc='lower center',
               bbox_to_anchor=(0.5, 0.0), ncol=ncol,
               framealpha=0.95, edgecolor='#CCCCCC', fontsize=9)
    plt.subplots_adjust(bottom=0.12)
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    return fig


def _leg_handles(with_ref=False):
    h = [
        mpatches.Patch(facecolor='#444444', hatch='////', edgecolor='#222222',
                       label='Mokymosi greitis 0,001'),
        mpatches.Patch(facecolor='#BBBBBB', hatch='',    edgecolor='#555555',
                       label='Mokymosi greitis 0,0005'),
    ]
    if with_ref:
        h.append(Line2D([0],[0], color=REF_COLOR, linestyle='--', linewidth=1.5,
                        label='Triukšmingas signalas prieš apdorojimą'))
    return h


def issaugoti(fig, vardas):
    fig.savefig(os.path.join(FIGURES_DIR, vardas+'.png'), dpi=300,
                bbox_inches='tight', facecolor='white')
    fig.savefig(os.path.join(FIGURES_DIR, vardas+'.svg'),
                bbox_inches='tight', facecolor='white')
    print(f'  ✓ {vardas}')

print('Pagalbinės funkcijos apibrėžtos.')

---
## 6. 14 pav. — PESQ (L2 ir L1)
**Failai:** `14a_modeliu_atranka_pesq_L2.png`, `14b_modeliu_atranka_pesq_L1.png`

In [ ]:
df_L2 = df_screen[df_screen['loss_name']=='L2']
df_L1 = df_screen[df_screen['loss_name']=='L1']

# 14 pav. a — PESQ L2
fig = plot_bar(df_L2, 'mean_pesq_enhanced',
    'Modelių atrankos PESQ rezultatai taikant L2 nuostolių funkciją',
    'PESQ reikšmė', fmt='pesq', ref_val=PESQ_REF)
issaugoti(fig, '14a_modeliu_atranka_pesq_L2'); plt.show()

# 14 pav. b — PESQ L1
fig = plot_bar(df_L1, 'mean_pesq_enhanced',
    'Modelių atrankos PESQ rezultatai taikant L1 nuostolių funkciją',
    'PESQ reikšmė', fmt='pesq', ref_val=PESQ_REF)
issaugoti(fig, '14b_modeliu_atranka_pesq_L1'); plt.show()

---
## 7. 15 pav. — STOI (L2 ir L1)
**Failai:** `15a_modeliu_atranka_stoi_L2.png`, `15b_modeliu_atranka_stoi_L1.png`

In [ ]:
# 15 pav. a — STOI L2
# Pastaba: STOI reikšmės telpa siaurame intervale (~0,895–0,910); x ašis pradedama nuo ~0,888
fig = plot_bar(df_L2, 'mean_stoi_enhanced',
    'Modelių atrankos STOI rezultatai taikant L2 nuostolių funkciją',
    'STOI reikšmė', fmt='stoi', ref_val=STOI_REF)
issaugoti(fig, '15a_modeliu_atranka_stoi_L2'); plt.show()

# 15 pav. b — STOI L1
fig = plot_bar(df_L1, 'mean_stoi_enhanced',
    'Modelių atrankos STOI rezultatai taikant L1 nuostolių funkciją',
    'STOI reikšmė', fmt='stoi', ref_val=STOI_REF)
issaugoti(fig, '15b_modeliu_atranka_stoi_L1'); plt.show()

---
## 8. 16 pav. — ΔSNR (L2 ir L1)
**Failai:** `16a_modeliu_atranka_delta_snr_L2.png`, `16b_modeliu_atranka_delta_snr_L1.png`

In [ ]:
# 16 pav. a — ΔSNR L2
fig = plot_bar(df_L2, 'mean_delta_snr_est',
    'Modelių atrankos ΔSNR rezultatai taikant L2 nuostolių funkciją',
    'ΔSNR, dB', fmt='delta')
issaugoti(fig, '16a_modeliu_atranka_delta_snr_L2'); plt.show()

# 16 pav. b — ΔSNR L1
fig = plot_bar(df_L1, 'mean_delta_snr_est',
    'Modelių atrankos ΔSNR rezultatai taikant L1 nuostolių funkciją',
    'ΔSNR, dB', fmt='delta')
issaugoti(fig, '16b_modeliu_atranka_delta_snr_L1'); plt.show()

---
## 9. 17 pav. — Validavimo nuostolis (L2 ir L1)
**Failai:** `17a_modeliu_atranka_validavimo_nuostolis_L2.png`, `17b_modeliu_atranka_validavimo_nuostolis_L1.png`

In [ ]:
# 17 pav. a — Validavimo nuostolis L2  (MSE)
# Pastaba: L1 (MAE) ir L2 (MSE) reikšmės skaičiuojamos skirtingomis formulėmis —
#          tiesioginis skaitinis palyginimas tarp L1 ir L2 nėra korektiškas.
fig = plot_bar(df_L2, 'best_val_loss',
    'Modelių atrankos validavimo nuostolis taikant L2 nuostolių funkciją',
    'Validavimo nuostolis', fmt='val', higher_is_better=False)
issaugoti(fig, '17a_modeliu_atranka_validavimo_nuostolis_L2'); plt.show()

# 17 pav. b — Validavimo nuostolis L1  (MAE)
fig = plot_bar(df_L1, 'best_val_loss',
    'Modelių atrankos validavimo nuostolis taikant L1 nuostolių funkciją',
    'Validavimo nuostolis', fmt='val', higher_is_better=False)
issaugoti(fig, '17b_modeliu_atranka_validavimo_nuostolis_L1'); plt.show()

---
## 10. CSV santraukos lentelių išsaugojimas

In [ ]:
santrauka = df_screen[[
    'etikete','model_name','loss_name','learning_rate','best_val_loss',
    'mean_pesq_noisy','mean_pesq_enhanced','mean_delta_pesq',
    'mean_stoi_noisy','mean_stoi_enhanced','mean_delta_stoi','mean_delta_snr_est',
]].copy()
santrauka.columns = [
    'Konfigūracija','Modelis','Nuostolių funkcija','Mokymosi greitis',
    'Validavimo nuostolis',
    'PESQ prieš','PESQ po','ΔPESQ',
    'STOI prieš','STOI po','ΔSTOI','ΔSNR',
]
santrauka = santrauka.sort_values('PESQ po', ascending=False).reset_index(drop=True)

p_san = os.path.join(FIGURES_DIR, 'modeliu_atranka_santrauka.csv')
santrauka.to_csv(p_san, index=True, encoding='utf-8-sig')
print(f'Santrauka: {p_san}  ({len(santrauka)} eilučių)')

p_top = os.path.join(FIGURES_DIR, 'modeliu_atranka_top5.csv')
santrauka.head(5).to_csv(p_top, index=True, encoding='utf-8-sig')
print(f'TOP 5: {p_top}')
display(santrauka.head(10))

---
## 12. Rezultatų failų patikra

In [ ]:
print('=== REZULTATŲ FAILŲ PATIKRA ===')
print(f'screen_lt_ konfigūracijų: {len(df_screen)}')
for ext in ['png','svg','csv']:
    ff = [f for f in sorted(os.listdir(FIGURES_DIR)) if f.endswith('.'+ext) and
          any(x in f for x in ['14a','14b','15a','15b','16a','16b','17a','17b'])]
    print(f'{ext.upper():3s} ({len(ff)}): {ff}')
print('\n=== TOP 5 pagal PESQ po apdorojimo ===')
display(santrauka[['Konfigūracija','PESQ po','STOI po','ΔSNR']].head(5))
print(f'Konfigūracijų su neigiamu ΔSNR: {(df_screen["mean_delta_snr_est"]<0).sum()}')